In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import os

csv_path = os.path.join("..", "data", "processed", "multi_ticker_options_clean.csv")
df = pd.read_csv(csv_path)

print("✅ Original dataset loaded:", df.shape)


✅ Original dataset loaded: (1869, 20)


In [2]:
df["date_collected"] = pd.to_datetime(df["date_collected"])
df["day_of_week"] = df["date_collected"].dt.dayofweek
df["month"] = df["date_collected"].dt.month


In [3]:
df["volume_oi_ratio"] = df["volume"] / (df["openInterest"] + 1)
df["bid_ask_spread"] = df["ask"] - df["bid"]
df["mid_price"] = (df["ask"] + df["bid"]) / 2
df["last_mid_diff"] = df["lastPrice"] - df["mid_price"]


In [4]:
def compute_greeks(S, K, T, r, sigma, option_type="call"):
    if sigma <= 0 or T <= 0:
        return 0.0, 0.0, 0.0, 0.0
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == "call":
        delta = norm.cdf(d1)
        theta = (- (S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * norm.cdf(d2)) / 365
    else:
        delta = -norm.cdf(-d1)
        theta = (- (S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T)) + r * K * np.exp(-r * T) * norm.cdf(-d2)) / 365
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    vega = S * norm.pdf(d1) * np.sqrt(T) / 100
    return delta, gamma, theta, vega

df["delta"], df["gamma"], df["theta"], df["vega"] = zip(*df.apply(lambda row: compute_greeks(
    row["spot_price"],
    row["strike"],
    row["days_to_expiry"] / 365,
    0.05,  # risk-free rate
    row["impliedVolatility"]
), axis=1))


In [5]:
# Create folder if it doesn’t exist
os.makedirs("../data/processed", exist_ok=True)

# Save the enriched dataset
df.to_csv("../data/processed/enriched_options_with_greeks.csv", index=False)
print("✅ Enriched dataset saved to ../data/processed/enriched_options_with_greeks.csv")


✅ Enriched dataset saved to ../data/processed/enriched_options_with_greeks.csv
